In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import obs
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42

### Setup

In [ ]:
# config
DATASET = "1461ka"
N_FEATURES = 4

# one config per insolation record: the choice is part of the config (and so of the run tag)
cfgs = {insol: rc.make_config(DATASET, seed=seed, params=None, insol=insol, test_frac=0.2)
        for insol in rc.INSOL_COLS}

cfg = rc.make_config(
    DATASET,
    seed=seed,
    params=None,         # None -> tuned params for DATASET; e.g. rc.DATASETS['1243ka'] to swap
    test_frac=0.2,       # pass test_size=461 instead to reproduce the old hard-coded split
)
paramsa = cfgs['insol_huy06']['params']
pathsa = rc.run_paths(cfgs['insol_huy06']['run_tag'])

print("run tag :", cfgs['insol_huy06']['run_tag'])
print("weights :", pathsa['weights'])
print("params  :", paramsa)

In [ ]:
# load in and split data
raw_huy06 = rc.load_data(cfgs['insol_huy06'])
raw_ber90 = rc.load_data(cfgs['insol_ber90'])
splits_huy06 = rc.split_data(raw_huy06, cfgs['insol_huy06'])
splits_ber90 = rc.split_data(raw_ber90, cfgs['insol_ber90'])

print(f'using huy06 record: {rc.describe_split(splits_huy06)}')
print(f'using ber90 record: {rc.describe_split(splits_ber90)}')

# time splits same for both insolation records, since they share the same age column
testt, traint, valt = splits_huy06['test_time'], splits_huy06['train_time'], splits_huy06['validation_time']


In [ ]:
# normalize data
scalea, traina, vala, testa = rc.fit_and_transform_splits(splits_huy06)
scaleb, trainb, valb, testb = rc.fit_and_transform_splits(splits_ber90)
# can also get and transform standard deviations of records

# chunk normalized data into windows
X_traina, Y_traina, Insol_traina = LPNE.create_windows(traina, cfgs['insol_huy06']['window_size'], cfgs['insol_huy06']['rollout_length'])
X_vala, Y_vala, Insol_vala = LPNE.create_windows(vala, cfgs['insol_huy06']['window_size'], cfgs['insol_huy06']['rollout_length_validation'])
X_trainb, Y_trainb, Insol_trainb = LPNE.create_windows(trainb, cfgs['insol_ber90']['window_size'], cfgs['insol_ber90']['rollout_length'])
X_valb, Y_valb, Insol_valb = LPNE.create_windows(valb, cfgs['insol_ber90']['window_size'], cfgs['insol_ber90']['rollout_length_validation'])

#print(X_traina)


### Training

##### using Huy06

In [ ]:
LPNE.set_seeds(cfgs['insol_huy06']['seed'])
braina = LPNE.CNN_model(cfgs['insol_huy06']['window_size'], params=paramsa, n_features=N_FEATURES)
braina.summary()

emua = AREmulator.RolloutEmulator(braina,
                                rollout_length=cfgs['insol_huy06']['rollout_length'],
                                vali_rollout_length=cfgs['insol_huy06']['rollout_length_validation'],
                                window_size=cfgs['insol_huy06']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=paramsa['dwt gamma'],
                                dwt_beta=paramsa['dwt beta'],
                                dwt_eps=paramsa['dwt eps'],
                                dwt_levels=paramsa['dwt levels'],
                                )

emua.noise_level = cfgs['insol_huy06']['noise_level'] # this can be done when the class is instantiated
emua.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=paramsa['learning rate'], clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['insol_huy06']['window_size'], N_FEATURES))
_ = emua(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['insol_huy06'], scalea, splits_huy06)

checka = sc.PhysicalAttractorCheckpoint(
    filepath = pathsa['weights'],
    acf_threshold = cfgs['insol_huy06']['acf_threshold']
)
loggera = sc.WeightStatsLogger()

history = emua.fit(
    [X_traina, Insol_traina],
    Y_traina,
    validation_data=([X_vala, Insol_vala], Y_vala),
    epochs=cfgs['insol_huy06']['epochs'],
    callbacks=[checka, loggera],
    batch_size=cfgs['insol_huy06']['batch_size'],
    validation_batch_size=cfgs['insol_huy06']['batch_size'],
    shuffle=True,
)


##### using Ber90

In [ ]:
paramsb = cfgs['insol_ber90']['params']
pathsb = rc.run_paths(cfgs['insol_ber90']['run_tag'])
print("run tag :", cfgs['insol_ber90']['run_tag'])

LPNE.set_seeds(cfgs['insol_ber90']['seed'])
brainb = LPNE.CNN_model(cfgs['insol_ber90']['window_size'], params=paramsb, n_features=N_FEATURES)

emub = AREmulator.RolloutEmulator(brainb,
                                rollout_length=cfgs['insol_ber90']['rollout_length'],
                                vali_rollout_length=cfgs['insol_ber90']['rollout_length_validation'],
                                window_size=cfgs['insol_ber90']['window_size'],
                                n_features=N_FEATURES,
                                dwt_gamma=paramsb['dwt gamma'],
                                dwt_beta=paramsb['dwt beta'],
                                dwt_eps=paramsb['dwt eps'],
                                dwt_levels=paramsb['dwt levels'],
                                noise_level=cfgs['insol_ber90']['noise_level'],
                                )
emub.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=paramsb['learning rate'], clipnorm=1.0))

dummy_X = tf.zeros((1, cfgs['insol_ber90']['window_size'], N_FEATURES))
_ = emub(dummy_X)

# creates runs/<run_tag>/ and writes config.json with the scaler used for this data
rc.save_run(cfgs['insol_ber90'], scaleb, splits_ber90)

checkb = sc.PhysicalAttractorCheckpoint(
    filepath = pathsb['weights'],
    acf_threshold = cfgs['insol_ber90']['acf_threshold']
)
loggerb = sc.WeightStatsLogger()

historyb = emub.fit(
    [X_trainb, Insol_trainb],
    Y_trainb,
    validation_data=([X_valb, Insol_valb], Y_valb),
    epochs=cfgs['insol_ber90']['epochs'],
    callbacks=[checkb, loggerb],
    batch_size=cfgs['insol_ber90']['batch_size'],
    validation_batch_size=cfgs['insol_ber90']['batch_size'],
    shuffle=True,
)

### Testing

In [ ]:
from unittest import mock

# boundary between the test period and the train/validation period (test ends at 1170 ka, train starts
# at 1168 ka; split_data skips the row in between), the same for both insolation records
TEST_END = (testt.min() + traint.max()) / 2

def plot_rollout_split(model, data, time, window_size, rollout_data, boundary=TEST_END):
    '''
        sp.plot_rollout, plus a vertical line at `boundary` [ka] on every panel. plot_rollout calls
        plt.show() itself, which would display (and close) the figure before we could draw on it, so that
        one call is held back with mock.patch, the line is added, and the figure is shown here instead.
        Returns plot_rollout's (overall_rmse, overall_r2, overall_ma_rmse).
    '''
    with mock.patch.object(plt, 'show'):
        metrics = sp.plot_rollout(model, data, time, window_size, rollout_data=rollout_data)
    fig = plt.gcf()
    for ax in fig.axes:
        ax.axvline(boundary, color='black', linestyle=':', lw=2)
    fig.axes[0].annotate('test  |  train / validation', xy=(boundary, 1.0), xycoords=('data', 'axes fraction'),
                         xytext=(0, 6), textcoords='offset points', ha='center', va='bottom', fontsize=14)
    plt.show()
    return metrics

##### using Huy06

In [ ]:
print(f"Loading best physical model for final evaluation from {pathsa['weights']} ...")
emua.load_weights(str(pathsa['weights']))

best_epoch_a = checka.best_epoch
print(f"Best physical model found at epoch {best_epoch_a} with ACF threshold {cfgs['insol_huy06']['acf_threshold']}.")


In [ ]:
sp.LossPlots(history, checka)

In [ ]:
w = cfgs['insol_huy06']['window_size']

# rollout over the test period only (1461-1170 ka)
rollout_a = sp.rollout(emua, testa, w)
sp.plot_rollout(emua, testa, testt, w, rollout_data = rollout_a)

# rollout seeded from the same oldest window, continued through train and validation to the present.
# Uses the full scaled record instead of joining the splits: `+` on numpy arrays adds them elementwise,
# and split_data leaves a 1-row gap between test and train, so joined splits would skip an age
fulla = scalea.transform(splits_huy06['full'])
fullt = splits_huy06['time']
rollout_full_a = sp.rollout(emua, fulla, w)
plot_rollout_split(emua, fulla, fullt, w, rollout_data = rollout_full_a)

##### using Ber90

In [ ]:
print(f"Loading best physical model for final evaluation from {pathsb['weights']} ...")
emub.load_weights(str(pathsb['weights']))

best_epoch_b = checkb.best_epoch
print(f"Best physical model found at epoch {best_epoch_b} with ACF threshold {cfgs['insol_ber90']['acf_threshold']}.")

In [ ]:
sp.LossPlots(historyb, checkb)

In [ ]:
w = cfgs['insol_ber90']['window_size']

# rollout over the test period only (1461-1170 ka)
rollout_b = sp.rollout(emub, testb, w)
sp.plot_rollout(emub, testb, testt, w, rollout_data = rollout_b)

# rollout seeded from the same oldest window, continued through train and validation to the present
fullb = scaleb.transform(splits_ber90['full'])
rollout_full_b = sp.rollout(emub, fullb, w)
plot_rollout_split(emub, fullb, fullt, w, rollout_data = rollout_full_b)